In [ ]:
# makers internal hackathon for group Return 224221;

# first code block to show data used (can change the last four lines to check)

# python libraries and stuff
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

rng = np.random.default_rng(67)     # pseudo rng
n = 1500                            # number of samples

confidence = rng.integers(1, 6, n)
accuracy = rng.integers(0, 101, n)
difficulty = rng.integers(1, 6, n)
study_time = rng.integers(15, 121, n)

labels = np.where(
    (accuracy < 50) | (confidence <= 2), 0,
    np.where(
        (accuracy >= 75) & (confidence >= 4)
        & (difficulty <= 3),
        2,  # Exam practice
        1   # Guided practice
    )
)

df = pd.DataFrame({
    "confidence": confidence,
    "accuracy": accuracy,
    "difficulty": difficulty,
    "study_time": study_time,
    "label": labels
})

print(df.head(10))
print("\nNumber of examples:", len(df))
print("\nLabel counts:")
print(df["label"].value_counts().sort_index())

In [ ]:

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    classification_report
)

# Pseudo RNG seed : 67
tf.keras.utils.set_random_seed(67)

# Features and labels
feature_names = [
    "confidence",
    "accuracy",
    "difficulty",
    "study_time"
]

X = df[feature_names].to_numpy(dtype=np.float32)
y = df["label"].to_numpy(dtype=np.int32)

# split train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=67,
    stratify=y
)

# put scalar on training data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

# neural network
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

# train
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    verbose=0
)

# Evaluate on held-out test set
test_loss, test_accuracy = model.evaluate(
    X_test, y_test, verbose=0
)

logits = model.predict(X_test, verbose=0)
predictions = np.argmax(logits, axis=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")
print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    labels=[0, 1, 2],
    target_names=[
        "Foundational Review",
        "Guided Practice",
        "Exam Practice"
    ],
    zero_division=0
))

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


cm = confusion_matrix(
    y_test, predictions, labels=[0, 1, 2]
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Review",
        "Guided",
        "Exam Practice"
    ]
).plot(cmap="Blues", values_format="d")

plt.title("Predicted vs Actual Labels")
plt.tight_layout()
plt.show()

model.save("studysmart_model.keras")
joblib.dump(scaler, "studysmart_scaler.joblib")

print("\nSaved model and scaler successfully.")
